# KadiPy Market - Module Pricing & Ingestion

Ce notebook démontre l'utilisation de `MarketPricing` et `WFPDataBridgesClient` pour l'ingestion, la normalisation et la détection d'anomalies sur les prix du marché.

---

## 1. Initialisation du Système

Nous commençons par instancier les classes principales. Nous créons d'abord un `client` WFP en forçant le mode `use_local_mirror=True`. 
Cela signifie : *"Ne cherche pas à te connecter à Internet, utilise le simulateur de données interne"*. 
Ensuite, on injecte ce client dans la classe mère `MarketPricing`.

**Pourquoi ?** Cela garantit que notre logique métier est prête et dispose d'une source de données valide pour travailler, sans risquer de planter pour des problèmes de réseau ou de clé API.

In [9]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

from kadi.market.data_ingestion import WFPDataBridgesClient
from kadi.market.pricing import MarketPricing

# 1. Initialisation avec le mode hors-ligne forcé (pour l'exemple sans clé API)
client = WFPDataBridgesClient(use_local_mirror=True)
pricing = MarketPricing(wfp_client=client)

print("Module Pricing initialisé avec succès !")

Module Pricing initialisé avec succès !


---
## 2. Récupération des données (`fetch_prices`)

La méthode `fetch_prices` est la porte d'entrée pour récupérer l'historique des marchés. Elle demande à la source de données (le WFP ou notre miroir local) de lui livrer les prix du maïs (`maize`) à Savalou sur les 30 derniers jours (`days_back=30`).

**Explication de la sortie :**
La sortie est un tableau (DataFrame pandas) qui possède une structure standardisée essentielle pour KadiPy :
- `date` : Le moment du relevé.
- `price` : La valeur numérique du prix sur le marché (ici, des prix générés aléatoirement autour d'une fourchette réaliste par notre miroir local).
- `unit` : Toujours formaté, ici `XOF/kg` (Francs CFA par kilo).

In [10]:
df_prices = pricing.fetch_prices(crop='maize', market='savalou_market', days_back=30)
display(df_prices.head())

,date,price,unit
0,2026-06-04,316.869539,XOF/kg
1,2026-06-05,311.501969,XOF/kg
2,2026-06-06,313.780115,XOF/kg
3,2026-06-07,259.468541,XOF/kg
4,2026-06-08,309.945521,XOF/kg


---
## 3. Normalisation des Unités (`normalize_units`)

Dans le monde agricole, les prix sont rapportés de manière chaotique : en Tonnes, en sacs de 100kg, ou au kilo. 
La méthode `normalize_units` prend un prix brut, lit son unité d'origine (`unit_orig`), et le ramène mathématiquement à une échelle standard universelle pour KadiPy (le kilogramme).

**Explication de la sortie :**
Le système reconnaît le mot "Tonne" et divise automatiquement le prix par 1000. C'est crucial : si notre IA de prévision mélangeait des prix au kilo et des prix à la tonne, elle produirait des prédictions totalement erronées.

In [11]:
prix_tonne = 250000.0
prix_kg = pricing.normalize_units(prix_tonne, unit_orig='XOF/Tonne', crop='maize')
print(f"{prix_tonne} XOF/Tonne = {prix_kg} XOF/kg")

250000.0 XOF/Tonne = 250.0 XOF/kg


---
## 4. Détection d'anomalies par Z-Score (`detect_anomalies`)

Les collecteurs de données sur le terrain font parfois des erreurs de frappe (ajouter un 0 de trop). 
La méthode `detect_anomalies` scanne l'historique complet en utilisant un algorithme statistique appelé **Z-Score**. Il calcule la moyenne des prix et l'écart-type (la volatilité). Si un point de donnée s'éloigne trop de la norme (généralement 3 fois l'écart-type), le système marque ce point comme suspect.

**Explication de la sortie :**
Dans la cellule ci-dessous, nous injectons volontairement un prix de 1500 XOF/kg (très élevé pour le maïs). 
Le tableau final n'affiche que cette ligne avec une nouvelle colonne `is_anomaly = True`. Le système a compris de lui-même que 1500 FCFA/kg pour du maïs est totalement incohérent vis-à-vis des autres prix. Le module d'Aide à la Décision ou de Prévision ignorera cette ligne biaisée grâce à ce tag.

In [12]:
# Ajout d'une anomalie évidente à la fin de notre DataFrame (1500 FCFA/kg)
df_prices.loc[30] = {'date': pd.Timestamp.today(), 'price': 1500.0, 'unit': 'XOF/kg'}

# Application de l'algorithme de détection
df_clean = pricing.detect_anomalies(df_prices)

# Affichage uniquement des anomalies trouvées
display(df_clean[df_clean['is_anomaly'] == True])

,date,price,unit,is_anomaly
30,2026-07-04 17:55:10.999639,1500.0,XOF/kg,True
